# Phase 01: Data Collection and Preprocessing

This notebook downloads market data for the **RegimeShift: Macro-Aware Tactical Asset Allocation Engine** project.

The goal of this phase is to create a clean and aligned dataset of asset prices and daily log returns that can be used in later phases for:

- Feature engineering
- HMM-based regime detection
- Portfolio optimization
- Walk-forward backtesting

The asset universe used here includes:

| Ticker | Meaning |
|---|---|
| `^NSEI` | NIFTY 50 index, used as the equity proxy |
| `IEF` | Intermediate-term bond ETF, used as the bond proxy |
| `GLD` | Gold ETF, used as the gold proxy |
| `^VIX` | Volatility index, used as a market stress indicator |


## 1. Imports and Folder Setup

This cell imports the required libraries and creates the required project folders if they do not already exist.

The notebook is written so that it works whether it is run from the project root folder or from inside the `notebooks/` folder.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from pathlib import Path
import os

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

if Path.cwd().name == "notebooks":
    DATA_DIR = Path("../data")
    OUTPUT_DIR = Path("../outputs")
else:
    DATA_DIR = Path("data")
    OUTPUT_DIR = Path("outputs")

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Imports completed")
print(f"Current working directory: {Path.cwd()}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 2. Project Configuration

The following tickers and date range are used for the project. The date range can be changed later if required.


In [ ]:
TICKERS = ["^NSEI", "IEF", "GLD", "^VIX"]
START_DATE = "2015-01-01"
END_DATE = "2024-01-01"

print(f"Tickers: {TICKERS}")
print(f"Date range: {START_DATE} to {END_DATE}")

## 3. Download Market Data

Daily OHLCV data is downloaded using `yfinance`.

`auto_adjust=True` is used so that the prices are adjusted for dividends and splits where applicable.


In [ ]:
print(f"Downloading {TICKERS} from {START_DATE} to {END_DATE}...")

raw_data = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    progress=False,
    auto_adjust=True
)

print(f"✅ Downloaded {len(raw_data)} trading days")
print(f"Raw data shape: {raw_data.shape}")
print(f"Column levels: {raw_data.columns.names}")

raw_data.head()

## 4. Extract Adjusted Close Prices

For this project, the adjusted close price series are used for return calculation and portfolio backtesting.


In [ ]:
close_prices = raw_data["Close"].copy()
close_prices.columns.name = None

close_prices = close_prices.rename(
    columns={
        "^NSEI": "NSE",
        "IEF": "IEF",
        "GLD": "GLD",
        "^VIX": "VIX"
    }
)

close_prices = close_prices[["NSE", "IEF", "GLD", "VIX"]]

print("✅ Close prices extracted")
print(close_prices.head())
print("\nMissing values:")
print(close_prices.isna().sum())

## 5. Handle Missing Values and Align Data

Different markets may have different holidays and missing dates.

The data is cleaned by dropping rows where any required asset price is missing. This ensures that all assets are aligned on the same trading dates.


In [ ]:
close_prices_clean = close_prices.dropna().copy()

print(f"Rows before cleaning: {len(close_prices)}")
print(f"Rows after cleaning:  {len(close_prices_clean)}")
print(f"Rows dropped:         {len(close_prices) - len(close_prices_clean)}")

close_prices_clean.head()

## 6. Compute Daily Log Returns

Log returns are used because they are additive over time and commonly used in quantitative finance.

The VIX is kept as a level because it is a volatility/stress index, not a traded asset in this setup.


In [ ]:
ASSET_COLS = ["NSE", "IEF", "GLD"]

asset_returns = np.log(close_prices_clean[ASSET_COLS]).diff()
vix_level = close_prices_clean["VIX"]

master_df = asset_returns.join(vix_level, how="inner").dropna()

print("✅ Master returns dataset created")
print(master_df.head())
print(f"\nFinal dataset shape: {master_df.shape}")

## 7. Save Processed Data

The cleaned price data and aligned return dataset are saved to the `data/` folder for use in later phases.


In [ ]:
master_df.to_csv(DATA_DIR / "master_returns.csv")
close_prices_clean.to_csv(DATA_DIR / "asset_prices.csv")

print(f"✅ Saved aligned returns to: {DATA_DIR / 'master_returns.csv'}")
print(f"✅ Saved cleaned prices to: {DATA_DIR / 'asset_prices.csv'}")

## 8. Basic Data Inspection

This section performs a quick sanity check on the return data before moving to feature engineering.


In [ ]:
print("Summary statistics for asset returns:")
display(master_df[ASSET_COLS].describe())

print("\nCorrelation matrix:")
display(master_df[ASSET_COLS].corr())

## 9. Plot Asset Prices

This plot shows the normalized price movement of the tradable assets.


In [ ]:
normalized_prices = close_prices_clean[ASSET_COLS] / close_prices_clean[ASSET_COLS].iloc[0]

ax = normalized_prices.plot(figsize=(12, 5))
ax.set_title("Normalized Asset Prices")
ax.set_ylabel("Growth of 1 Unit")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

## 10. Plot Cumulative Returns

The cumulative return chart helps compare how different asset classes behaved over the sample period.


In [ ]:
cumulative_returns = np.exp(master_df[ASSET_COLS].cumsum())

ax = cumulative_returns.plot(figsize=(12, 5))
ax.set_title("Cumulative Returns of Tradable Assets")
ax.set_ylabel("Growth of 1 Unit")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

## 11. Plot VIX / Market Stress Indicator

The VIX series is used as a market stress indicator.

High VIX values generally correspond to more uncertain or volatile market conditions.


In [ ]:
ax = close_prices_clean["VIX"].plot(figsize=(12, 5), color="black")
ax.set_title("VIX Level")
ax.set_ylabel("VIX")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

## 12. Phase 1 Output Summary

This phase generated the following files:

| File | Description |
|---|---|
| `data/asset_prices.csv` | Cleaned adjusted close prices |
| `data/master_returns.csv` | Aligned daily log returns for assets plus VIX level |

These files will be used in Phase 2 for feature engineering.


In [ ]:
print("✅ Phase 1 completed successfully.")
print("Generated files:")
print(f"- {DATA_DIR / 'asset_prices.csv'}")
print(f"- {DATA_DIR / 'master_returns.csv'}")